# Generación Paquete de Datos

En este cuaderno generaremos lo necesario para empaquetar aquellos Datasets que se quieran ofertar en el Espacio de Datos CitriData/DSagro de la Universidad de Córdoba.

Para comenzar se necesita:
- **schema.json** generado para el SDM.
- Los datos a ofertar, ya sea en formato csv o json.
- La siguiente estructura de carpetas:

```text
../nombrePaquete/
	|
	|- data/
	|    |-data.csv (o .json)
	|    |-data.parquet [se genera en el proceso]
	|
	|- doc/
	|    |-rsc/
	|    |  |- *
	|    |
	|    |-doc.md
	|    |-doc.pdf
	|
	|- schema.json
	|- metadata_final.json [se genera en el proceso]
```

### Explicación estructura.
* **data**: Carpeta donde se guardaran los datos en el formato original (.csv, .json...) y en formato comprimido .parquet que se obtendrá con el script correspondiente.
* **doc**: Carpeta con toda la documentación e información acerca de los datos, sus fuentes de origen, observaciones, la metodología usada y los articulos/recursos que acompañen dicha documentación. Se escribirá la documentación en formato .md y en .pdf.
* **schema.json**: se trata del schema generado siguiendo el protocolo de los Smart Data Model (mismo que para el SDM).
* **metadata_final.json**: archivo de metadatos con información relativa al paquete de datos para el Data Space (connector id, provider, dataset,...) es generado con el script _generar_metadata.py_

### Dependencias del cuaderno

Para poder ejecutar este cuaderno solo necesita las siguientes dependencias:
- pandas
- pyarrow

Copie y pegue el siguiente comando en terminal para instalar el paquete. (Según su gestor de paquetes)

```bash
pip install pandas pyarrow
```

```bash
conda install -c conda-forge pandas pyarrow
```

## generar_parquet.py

En este apartado se pasarán los datos a formato .parquet, solo debe modificar la variable `PATH_IN` y las lineas de codigo que realizan la lectura inicial de los datos `df = pd.read...` .

In [ ]:
import pandas as pd
from pathlib import Path

# Path de los datos de entrada
PATH_IN = Path('AgriParcelNewPlantation/data/AgriParcelNewPlantation.json')

# Comentar/Descomentar según el tipo de archivo de entrada
df = pd.read_csv(PATH_IN, sep=',', encoding='latin-1') # si tu csv tiene otro separador, cámbialo en el parámetro sep. Por ejemplo, si es tabulador, sep='\t'
#df = pd.read_json(PATH_IN)

# Path de los datos de salida, modificar si lo quiere guardar en otra carpeta
PATH_OUT = PATH_IN.with_suffix('.parquet')

# [OPCIONAL] Eliminar columnas que no se quieren guardar en el archivo parquet
df.drop(columns=['cColor', 'cDeep'], inplace=True)

df.to_parquet(
    PATH_OUT,
    engine="pyarrow",
    compression="snappy",
    index=False
)

print("¡Conversión completada con éxito!")


In [ ]:
# Mostramos las primeras filas del DataFrame para compararlo con el parquet generado posteriormente
df.head()

,entity_id,idClasifica,idCobertura,cambioUso,dataOwner,dataProvider
0,ES.61.14.49.32.51.4,Plantacion Adulta,Primera,No,CitriData (c) 2025,Plataforma AgroFIWARE
1,ES.61.14.49.12.83.1,Plantacion Adulta,Primera,No,CitriData (c) 2025,Plataforma AgroFIWARE
2,ES.61.14.49.5.206.2,Plantacion Adulta,Primera,No,CitriData (c) 2025,Plataforma AgroFIWARE
3,ES.61.14.49.25.168.1,Plantacion Adulta,Primera,No,CitriData (c) 2025,Plataforma AgroFIWARE
4,ES.61.14.49.21.208.2,Plantacion Adulta,Primera,No,CitriData (c) 2025,Plataforma AgroFIWARE
...,...,...,...,...,...,...
1660,ES.61.14.49.26.3.1,Plantacion Adulta,Primera,No,CitriData (c) 2025,Plataforma AgroFIWARE
1661,ES.61.14.49.3.17.1,Plantacion Adulta,Primera,No,CitriData (c) 2025,Plataforma AgroFIWARE
1662,ES.61.14.49.10.4.1,Plantacion Adulta,Primera,No,CitriData (c) 2025,Plataforma AgroFIWARE
1663,ES.61.14.49.6.32.1,Plantacion Adulta,Primera,No,CitriData (c) 2025,Plataforma AgroFIWARE


In [23]:
# leemos el archivo Parquet para verificar que se haya guardado correctamente
df_parquet = pd.read_parquet(PATH_OUT, engine="pyarrow")
df_parquet.head()  # Muestra las primeras filas del DataFrame cargado desde Parquet

,entity_id,idClasifica,idCobertura,cambioUso,dataOwner,dataProvider
0,ES.61.14.49.32.51.4,Plantacion Adulta,Primera,No,CitriData (c) 2025,Plataforma AgroFIWARE
1,ES.61.14.49.12.83.1,Plantacion Adulta,Primera,No,CitriData (c) 2025,Plataforma AgroFIWARE
2,ES.61.14.49.5.206.2,Plantacion Adulta,Primera,No,CitriData (c) 2025,Plataforma AgroFIWARE
3,ES.61.14.49.25.168.1,Plantacion Adulta,Primera,No,CitriData (c) 2025,Plataforma AgroFIWARE
4,ES.61.14.49.21.208.2,Plantacion Adulta,Primera,No,CitriData (c) 2025,Plataforma AgroFIWARE


## generar_metadata.py

El siguiente paso es generar los metadatos que adjuntaremos en el paquete de datos, estos metadatos tienen informacion relativa a:
- Proveedor del dato
    - Información relativa al Espacio de Datos sobre el proveedor
- Fuente de datos original
- Variables del dataset

Para empezar este proceso debemos tener:
- schema.json (Generado para Smart Data Model del paquete a ofertar)
- data.parquet (Generado en el paso anterior)

### Configuración
Para generar el archivo _metadata_final.json_ solo necesita rellenar:
- **0.CONFIGURACIÓN FUENTE DE DATOS Y METADATA DEL DATASET**: Se trata de la información básica relativa a la fuente de datos original y a los datos.
- **1.CONFIGURACIÓN SOBERANA**: Información relativa al proveedor de datos, si no conoce sus datos de referencia en el Espacio de Datos puede comprobarlos en el siguiente enlace: [DSAgro - Registro](https://portal.dsagro.uco.es/registry/)

Tras rellenar las variables de los apartados descritos solo tiene que ejecutar la siguiente celda.

In [ ]:
import json
import hashlib
from copy import deepcopy
from datetime import datetime, timezone
import pyarrow.parquet as pq
import pandas as pd

# =====================================================================
# 0. CONFIGURACIÓN FUENTE DE DATOS Y METADATA DEL DATASET
# =====================================================================

#METADATA DEL DATASET
PACK_NAME = "InternationalCitrusTrade"  # Nombre del paquete de datos, usado en el manifiesto
PACK_TITLE = "Smart Data Models - InternationalCitrusTrade"  # Título del dataset, usado en el manifiesto
PACK_DESCRIPTION = "Data model for international trade statistics of citrus products, based on sources such as UN Comtrade and aligned with global trade data standards. It captures key variables including origin/destination countries, HS codes, trade values, volumes, and time periods. Designed to support consistent analysis, comparison, and integration with economic data systems."  # Descripción del dataset, usado en el manifiesto

#METADATA ORIGINAL
RUTA_ESCHEMA = "InternationalCitrusTrade/schema.json"  # Ruta del archivo de esquema de la plataforma
RUTA_PARQUET = "InternationalCitrusTrade/data/InternationalCitrusTrade.parquet"  # Ruta del archivo Parquet de origen
COLUMNAS_CLAVE = ["refPeriodId"]  # Columnas clave para el dataset, VARIABLE TEMPORAL EN POSICION [0] (dateCreate) // NO METER ENTITY_ID

#FUENTE DE DATOS ORIGINAL
SOURCE_NAME = "United Nations Comtrade database (UNComtrade)"  # Fuente de datos original, usado en el manifiesto
SOURCE_DATASET = "Acceso vía API"  # Nombre del dataset original, usado en el manifiesto
SOURCE_LICENSE = "https://uncomtrade.org/docs/policy-on-use-and-re-dissemination/"  # Licencia de uso de los datos originales, usado en el manifiesto
SOURCE_URL = "https://comtradeplus.un.org/TradeFlow"  # URL de la fuente de datos original, usado en el manifiesto

# Poner todas las fuentes: 
    # ORIGINALES (Ejemplo SiAR, UNComtrade, etc.)
    # PROPIAS que sean distintas al proveedor, se puede definir como la entidad que procesa los datos antes de ofertarlos (Ejemplo TEF (para caso Mildiu), HIBA, etc.) 
FUENTES_ORIGEN = [
    # {
    # ----EJEMPLO OTRA FUENTE QUE NO ES LA ORIGINAL, SINO QUE PROCESA LOS DATOS ANTES DE OFERTARLOS, en este caso son datos de un modelo predictivo que usa los datos SiAR
    #     "source": "AgrifoodTEF - Universidad de Córdoba",
    #     "dataset": "TEF - Mildiu Data v3",
    #     "license_name": "CC BY 4.0 ES",
    #     "url": "https://www.uco.es/agrifoodTEF"
    # },
    {
        "source": SOURCE_NAME,
        "dataset": SOURCE_DATASET,
        "license_name": SOURCE_LICENSE,
        "url": SOURCE_URL
    }
]
# =====================================================================
# 1. CONFIGURACIÓN SOBERANA (Pon aquí los datos de tu organización)
# =====================================================================
DATOS_PROVEEDOR = {
    "participant_id": "citridata-connector",
    "entity_name": "Espacio de Datos Federados de Agricultura de Precisión",
    "data_owner": "Proyecto CitriData",
    "connector_endpoint": "https://citridata.dsagro.uco.es/api/dsp",
    "connector_version": "0.1.4",
    "owner_website": "https://www.uco.es/citridata",
    "technical_operator": "Aula de Transformación Digital FIWARE",
    "operator_website": "https://www.uco.es/atdfiware",
    "did": "did:web:citridata.dsagro.uco.es",
    "trust_issuer": "did:web:portal.ds.uco.es",
    "connector_description": "Conector EDC para Citridata en el espacio de datos UCO"
}


#------- NO MODIFICAR-------: Diccionario para traducir los tipos técnicos a tipos lógicos amigables para el catálogo
MAPEO_LOGICO = {
    "int64": "integer",
    "int32": "integer",
    "double": "numeric",
    "float64": "numeric",
    "bool": "boolean",
    "string": "categorical_text",
    "object": "categorical_text"
}



# =====================================================================
# 2. FUNCIONES PARA PROCESAMIENTO Y EXTRACCIÓN DEL PARQUET
# =====================================================================
# -------NO MODIFICAR-------
def _extraer_properties(esquema_labels):
    """Extrae el dict de properties del schema, buscando en la raíz y dentro de allOf."""
    properties = esquema_labels.get("properties", {})
    if not properties:
        for item in esquema_labels.get("allOf", []):
            if "properties" in item:
                properties.update(item["properties"])
    return properties


def generar_metadata_final(ruta_metadata_input="metadata.json", ruta_schema_input=RUTA_ESCHEMA, ruta_json_output="metadata_final.json"):
    print(f"Leyendo metadata base de: {ruta_metadata_input}...")
    with open(ruta_metadata_input, "r", encoding="utf-8") as f_meta:
        metadata_base = json.load(f_meta)

    print(f"Leyendo esquema de etiquetas desde: {ruta_schema_input}...")
    with open(ruta_schema_input, "r", encoding="utf-8") as f_schema:
        esquema_labels = json.load(f_schema)

    etiquetas_por_nombre = {}
    tipos_por_nombre = {}
    descripciones_por_nombre = {}
    for nombre, definicion in _extraer_properties(esquema_labels).items():
        etiqueta = definicion.get("ccoc-label", {}).get("default")
        if etiqueta is not None:
            etiquetas_por_nombre[nombre] = etiqueta
        tipo_campo = definicion.get("type")
        if tipo_campo is not None:
            tipos_por_nombre[nombre] = tipo_campo
        descripcion_campo = definicion.get("description")
        if descripcion_campo is not None:
            descripciones_por_nombre[nombre] = descripcion_campo

    metadata_final = deepcopy(metadata_base)
    fields = metadata_final.get("dataset", {}).get("fields", [])
    for field in fields:
        nombre_campo = field.get("name")
        etiqueta = etiquetas_por_nombre.get(nombre_campo)
        description = descripciones_por_nombre.get(nombre_campo, field.get("description"))
        print(f"Procesando campo: {nombre_campo}, descripción original: {description}, etiqueta encontrada: {etiqueta}")
        if description is not None:
            field["description"] = description
        tipo_campo = tipos_por_nombre.get(nombre_campo)
        if tipo_campo is not None:
            field["logical_type"] = tipo_campo

    with open(ruta_json_output, "w", encoding="utf-8") as f_out:
        json.dump(metadata_final, f_out, indent=2, ensure_ascii=False)

    print(f"Archivo generado con éxito en '{ruta_json_output}'.")

def extraer_y_generar_json(ruta_parquet_input, ruta_json_output="metadata.json"):
    print(f"Leyendo metadatos binarios de: {ruta_parquet_input}...")
    
    # Leer el Footer del Parquet (Metadatos técnicos sin cargar filas)
    archivo_parquet = pq.ParquetFile(ruta_parquet_input)
    meta_tecnica = archivo_parquet.metadata
    esquema = archivo_parquet.schema
    
    # Calcular el Hash SHA-256 real del archivo para garantizar la integridad en el Dataspace
    print("Calculando hash criptográfico SHA-256...")
    sha256_hash = hashlib.sha256()
    with open(ruta_parquet_input, "rb") as f:
        for byte_block in iter(lambda: f.read(4096), b""):
            sha256_hash.update(byte_block)
    hash_final = sha256_hash.hexdigest()

    # Mapear dinámicamente las columnas y sus tipos de datos
    campos_esquema = []
    lista_nombres_columnas = []
    
    for i in range(len(esquema)):
        columna = esquema.column(i)
        tipo_fisico = str(columna.physical_type).lower()
        lista_nombres_columnas.append(columna.name)
        
        campos_esquema.append({
            "name": columna.name,
            "physical_type": tipo_fisico,
            "logical_type": MAPEO_LOGICO.get(tipo_fisico, "text"),
            "nullable": True,  # Parquet admite nulos por defecto en la mayoría de configuraciones
            "description": f"Campo {columna.name} extraído automáticamente del archivo Parquet.",
            "declared_type": tipo_fisico
        })

    # Cargar columnas clave puntuales para extraer estadísticas de negocio rápidamente
    print("Calculando estadísticas del dataset...")
    columna_identificador = next((col for col in ["entity_id", "id"] if col in lista_nombres_columnas), None)
    columnas_clave = [col for col in [columna_identificador] + COLUMNAS_CLAVE if col is not None and col in lista_nombres_columnas]
    
    # Leemos solo esas columnas para optimizar la RAM
    if columnas_clave:
        df_mini = pd.read_parquet(ruta_parquet_input, columns=columnas_clave)
    else:
        df_mini = pd.DataFrame()
    
    # Extraer métricas si las columnas existen en tu archivo
    year_min = None
    year_max = None

    
    # TODO: Normalizar la columna fecha para que tenga siempre el mismo nombre (EJ: dateCreate) y en el mismo formato, para poder automatizar la
    # extracción de años mínimos y máximos. Por ahora, se hace un chequeo condicional.
    if "dateCreate" in df_mini.columns:
        fechas = pd.to_datetime(df_mini["dateCreate"], errors="coerce", utc=True)
        years = fechas.dt.year.dropna()
        if not years.empty:
            year_min = int(years.min())
            year_max = int(years.max())


    entity_id_count = int(df_mini[columna_identificador].nunique()) if columna_identificador is not None and columna_identificador in df_mini.columns else None
    for col in COLUMNAS_CLAVE:
        if col in df_mini.columns:
            locals()[f"{col}_count"] = int(df_mini[col].nunique())
            break
    
    
    status_counts = {}
    if "temperature" in df_mini:
        # Convertir los conteos de filas a enteros nativos de Python para que JSON los acepte
        status_counts = {k: int(v) for k, v in df_mini["temperature"].value_counts().to_dict().items()}

    # =====================================================================
    # 3. CONSTRUCCIÓN ESTRUCTURA FINAL (MANIFIESTO)
    # =====================================================================
    manifiesto_completo = {
        "package_name": PACK_NAME,
        "dataset_version": "0.1.0",
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "provider": DATOS_PROVEEDOR,
        "dataset": {
            "title": PACK_TITLE,
            "description": PACK_DESCRIPTION,
            "file_name": ruta_parquet_input,
            "grain": f"{columna_identificador or 'id'}-" + COLUMNAS_CLAVE[0] if COLUMNAS_CLAVE else None,
            "primary_keys": [columna_identificador or "id", COLUMNAS_CLAVE[0]] if COLUMNAS_CLAVE else None,
            "rows": meta_tecnica.num_rows,
            "columns": lista_nombres_columnas,
            "column_count": meta_tecnica.num_columns,
            "year_min": year_min,
            "year_max": year_max,
            "entity_id_count": entity_id_count,
            #añadimos el conteo de las columnas clave
            **{f"{col}_count": locals().get(f"{col}_count") for col in COLUMNAS_CLAVE if locals().get(f"{col}_count") is not None},
            #"fecha_exploracion_count": fecha_exploracion_count,
            "sha256": hash_final,
            "fields": campos_esquema
        },
        "sources": FUENTES_ORIGEN,
        # "schema_reference_document": "data_contract.md",
        # "publication_policy": {
        #     "weather_gap_policy_v1": "keep_missing_without_imputation",
        #     "weather_gap_control_fields": ["weather_record_available", "weather_join_status"]
        # }
    }

    # Guardar en disco como archivo JSON legible
    with open(ruta_json_output, "w", encoding="utf-8") as f_out:
        json.dump(manifiesto_completo, f_out, indent=2, ensure_ascii=False)
        
    print(f"🌍 ¡Metadata intermedia generada con éxito en '{ruta_json_output}'!")

# =====================================================================
# 3. EJECUCIÓN DEL SCRIPT
# =====================================================================
if __name__ == "__main__":
    ruta_parquet = RUTA_PARQUET
    ruta_metadata_base = PACK_NAME + "/metadata.json"
    ruta_metadata_final = PACK_NAME + "/metadata_final.json"

    extraer_y_generar_json(ruta_parquet, ruta_metadata_base)
    generar_metadata_final(ruta_metadata_base, RUTA_ESCHEMA, ruta_metadata_final)


Leyendo metadatos binarios de: InternationalCitrusTrade/data/InternationalCitrusTrade.parquet...
Calculando hash criptográfico SHA-256...
Calculando estadísticas del dataset...
🌍 ¡Contrato de datos generado con éxito en 'InternationalCitrusTrade/metadata.json'!
Leyendo metadata base de: InternationalCitrusTrade/metadata.json...
Leyendo esquema de etiquetas desde: InternationalCitrusTrade/schema.json...
Procesando campo: entity_id, descripción original: Campo entity_id extraído automáticamente del archivo Parquet., etiqueta encontrada: None
Procesando campo: cmdDesc, descripción original: Property. Model:'https://schema.org/Text'. Units: 'Products'. Commodity description, etiqueta encontrada: None
Procesando campo: flowDesc, descripción original: Property. Model:'https://schema.org/Text'. Units: 'Flow'. Description of the trade flow, etiqueta encontrada: None
Procesando campo: reporterDesc, descripción original: Property. Model:'https://schema.org/Text'. Units: 'Country'. Country or reg